In [13]:
import os
import pandas as pd
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold 
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

In [2]:
# Se cargan los datos. 

data=pd.read_csv('car_details_v3.csv',sep=',')

In [3]:
data.shape

(8128, 13)

In [4]:
# Mostrar los datos

data.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)",5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.0 kmpl,1396 CC,90 bhp,22.4 kgm at 1750-2750rpm,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.1 kmpl,1298 CC,88.2 bhp,"11.5@ 4,500(kgm@ rpm)",5.0


In [5]:
# Es recomendable que todos los pasos de preparación se realicen sobre otro archivo.

data_t = data

data_t=data_t.dropna()
data_t=data_t.drop_duplicates()

data_t['mileage'] = data_t['mileage'].astype(str).str.extract(r'([\d.]+)').astype(float)
data_t['mileage'] = pd.to_numeric(data_t['mileage'], errors='coerce')
data_t['engine'] = data_t['engine'].astype(str).str.extract(r'([\d.]+)').astype(float)
data_t['engine'] = pd.to_numeric(data_t['engine'], errors='coerce')
data_t['max_power'] = data_t['max_power'].astype(str).str.extract(r'([\d.]+)').astype(float)
data_t['max_power'] = pd.to_numeric(data_t['max_power'], errors='coerce')

data_t.shape
data_t.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.40,1248.0,74.00,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14,1498.0,103.52,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.70,1497.0,78.00,"12.7@ 2,700(kgm@ rpm)",5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.00,1396.0,90.00,22.4 kgm at 1750-2750rpm,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.10,1298.0,88.20,"11.5@ 4,500(kgm@ rpm)",5.0


In [6]:
# El tamaño de los datos preparados

data_t.shape

(6717, 13)

In [17]:
data_t = pd.get_dummies(data_t, columns=['fuel','owner'], dtype=int)

#data_t=data_t.drop(['name'], axis=1)
#data_t=data_t.drop(['seller_type'], axis=1)
#data_t=data_t.drop(['transmission'], axis=1)
#data_t=data_t.drop(['torque'], axis=1)

KeyError: "None of [Index(['fuel', 'owner'], dtype='object')] are in the [columns]"

In [18]:
# Se selecciona la variable objetivo, en este caso "precio".

Y=data_t['selling_price']
X=data_t.drop(['selling_price'], axis=1)

In [19]:
# Se realiza la división entrenamiento - test. Se deja 20% de los datos para el test.

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.20, random_state = 0)

In [20]:
# Para acelerar la convergencia del algoritmo que utiliza Lasso para optimizar la función de costo, utilizaremos la opción
# de normalizar los datos para que todos estén en el mismo rango.

modelo_lasso = make_pipeline(StandardScaler(), Lasso(alpha=1))
modelo_lasso

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('lasso', Lasso(alpha=1))])

In [21]:
# Ajuste del modelo

modelo_lasso.fit(X_train,Y_train)

C:\Users\josep\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.846e+12, tolerance: 1.410e+11
  model = cd_fast.enet_coordinate_descent(


Pipeline(steps=[('standardscaler', StandardScaler()),
                ('lasso', Lasso(alpha=1))])

In [23]:
# Ahora probemos el rendimiento sobre el conjunto test.
y_pred = modelo_lasso.predict(X_test)

print("RMSE: %.2f" % mean_squared_error(Y_test, y_pred))
print("MAE: %.2f" % mean_absolute_error(Y_test, y_pred))
print('R²: %.2f' % r2_score(Y_test, y_pred))

RMSE: 129372456734.90
MAE: 181378.43
R²: 0.60


In [24]:
# Fijemos el número de particiones. Utilizaremos K = 10.

particiones = KFold(n_splits=10, shuffle=True, random_state = 0)

In [29]:
# Ahora tenemos que definir el espacio de búsqueda, es decir, los valores de alpha que queremos que sean considerados. 
# Para esto se define un diccionario (o grilla) con los valores que podrá asumir el hiperparámetro alpha.
# Probemos con los siguientes valores:

param_grid = {'lasso__alpha': [1, 2, 3, 4, 5]}

In [30]:
# Definimos el modelo sin ningún valor del hiperparámetro alpha

modelo_lasso = make_pipeline(StandardScaler(),Lasso())

In [31]:
# Ahora utilizamos GridSearch sobre el grid definido y con 10 particiones en la validación cruzada.

mejor_modelo = GridSearchCV(modelo_lasso, param_grid, cv=particiones, n_jobs=-1)
mejor_modelo.fit(X_train, Y_train)

C:\Users\josep\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.847e+12, tolerance: 1.410e+11
  model = cd_fast.enet_coordinate_descent(


GridSearchCV(cv=KFold(n_splits=10, random_state=0, shuffle=True),
             estimator=Pipeline(steps=[('standardscaler', StandardScaler()),
                                       ('lasso', Lasso())]),
             n_jobs=-1, param_grid={'lasso__alpha': [1, 2, 3, 4, 5]})

In [32]:
# Podemos ver cual fue el resultado de la búsqueda (mejor valor de alpha)

print("Mejor parámetro: {}".format(mejor_modelo.best_params_)) 

Mejor parámetro: {'lasso__alpha': 5}


In [33]:
# También puedes indicarle a GridSearch que seleccione el mejor modelo a partir de la búsqueda con base en una métrica 
# particular.

mejor_modelo = GridSearchCV(modelo_lasso, param_grid, scoring = 'neg_mean_absolute_error', n_jobs=-1 )
mejor_modelo.fit(X_train, Y_train)

print("Mejor parámetro: {}".format(mejor_modelo.best_params_)) 

Mejor parámetro: {'lasso__alpha': 5}


C:\Users\josep\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.847e+12, tolerance: 1.410e+11
  model = cd_fast.enet_coordinate_descent(


In [34]:
modelo_final = mejor_modelo.best_estimator_
y_pred = modelo_final.predict(X_test)

print("RMSE: %.2f" % mean_squared_error(Y_test, y_pred))
print("MAE: %.2f" % mean_absolute_error(Y_test, y_pred))
print('R²: %.2f' % r2_score(Y_test, y_pred))

RMSE: 129372365683.47
MAE: 181376.53
R²: 0.60


In [36]:
# Revisar los parámetros del modelo entrenado

coeficientes = mejor_modelo.best_estimator_.named_steps['lasso'].coef_
variables = X_train.columns

pd.DataFrame({'coeficientes':coeficientes,'variables':variables})

,coeficientes,variables
0,128499.315106,year
1,-59512.282116,km_driven
2,12307.047340,mileage
3,42058.467093,engine
4,302943.964199,max_power
5,-14828.486589,seats
6,1617.924108,fuel_CNG
7,26959.504688,fuel_Diesel
8,10497.918955,fuel_LPG
9,-2990.662897,fuel_Petrol
